**Importing Libraries and Models**

In [13]:
#Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
#preprocessing Imports
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [15]:
## Evaluation Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

In [16]:
# Models
from sklearn.linear_model import LogisticRegression,LassoCV,RidgeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

**Importing The Dataset:**

In [36]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 

**Dataset Preparation:**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# data (as pandas dataframes) 

X = bank_marketing.data.features
Y = bank_marketing.data.targets

label_encoder = LabelEncoder()
Y = LabelEncoder().fit_transform(np.ravel(Y))


print(f"Feature matrix shape : {X.shape}")
print(f"Target shape         : {Y.shape}")

**Display First Five Records**

In [ ]:
X.head()


**Train-Test Split:**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Data Split Summary
- Training Set: 80%
- Testing Set: 20%
- Random State: 42

**Identify Feature Types:**

In [ ]:
#Numerical Features
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

#Categorical Features
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical Features ({len(numerical_features)}):")
print(numerical_features)

print(f"\nCategorical Features ({len(categorical_features)}):")
print(categorical_features)

**Missing Value Handling**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
#Numerical Pipeline
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)
#Categorical Pipeline
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

#Combine Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

**Evaluate Function to Give All Metrics After Model Training**

In [51]:
#for xgboost, we need to convert the data into DMatrix format
from sklearn.preprocessing import LabelEncoder

def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on the test set and print various metrics.
    Parameters:
    model: Trained model to evaluate
    X_test: Test features
    y_test: True labels for the test set
    """
    if model.__class__.__name__  is XGBClassifier:
        label_encoder = LabelEncoder()
        y_test_encoded = label_encoder.fit_transform(y_test)


    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    
    # Print metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC AUC Score: {roc_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

**Testing Models**

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "SVC": SVC(probability=True),
    "KNeighborsClassifier": KNeighborsClassifier(),
    "GaussianNB": GaussianNB(),
    "CatBoostClassifier": CatBoostClassifier(verbose=0),
    "XGBClassifier": XGBClassifier(eval_metric='logloss')
}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    evaluate_model(model, X_test, y_test)
    print("\n" + "="*50 + "\n")

**Best Model is CatBoostClassifier**

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search_params = {
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "iterations": [500, 1000]
}
grid = GridSearchCV(
    estimator=CatBoostClassifier(verbose=0,random_state=42),
    param_grid=grid_search_params,
    cv=2,
    scoring="accuracy"
)
grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
#metics for the best model
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


In [ ]:
 # Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

**Prediction:**

In [ ]:
prediction_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

prediction_df

**Conclusion:**
-Successfully preprocessed the dataset by handling missing values, encoding categorical features, and scaling numerical features.

-Trained and evaluated multiple classification models, including Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, 
 AdaBoost,  XGBoost, CatBoost, and SVM.

-Compared model performance using Accuracy, Precision, Recall, F1-score, ROC-AUC, and Confusion Matrix.

-CatBoost achieved the best overall performance on the dataset after hyperparameter tuning.

-The dataset is imbalanced, with significantly more "No" responses than "Yes" responses; therefore, accuracy alone is not sufficient for 
 evaluating model performance.

-The confusion matrix showed that the model correctly classified most "No" instances, but improving the identification of "Yes" instances 
 by reducing false negatives remains a key area for improvement.

-Overall, the trained model provides a strong baseline for predicting whether a customer will subscribe to a term deposit.